# Ejemplo de uso del modulo con dataset del sistema ADATRAP

In [1]:
from pathlib import Path
import sys

REPO_ROOT = Path("../../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

DATA_PATH = REPO_ROOT / "data" / "ADATRAP" 

In [2]:
import pandas as pd
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

In [107]:
from __future__ import annotations

from pathlib import Path
from typing import Any, Optional

import re
import unicodedata

import numpy as np
import pandas as pd
from pyproj import Transformer

try:
    import yaml
except Exception:
    yaml = None


# -----------------------------------------------------------------------------
# Configuración base
# -----------------------------------------------------------------------------

DEFAULT_SOURCE_CRS = "EPSG:5361"
DEFAULT_TARGET_CRS = "EPSG:4326"

NULL_LIKE_VALUES = {
    "",
    "-",
    "--",
    "nan",
    "none",
    "null",
    "na",
    "n/a",
}


# -----------------------------------------------------------------------------
# Helpers de lectura / YAML
# -----------------------------------------------------------------------------

def load_yaml_file(path: str | Path) -> dict[str, Any]:
    if yaml is None:
        raise ImportError("pyyaml no está instalado. Instálalo para cargar YAML.")
    path = Path(path)
    with open(path, "r", encoding="utf-8") as f:
        data = yaml.safe_load(f)
    return data or {}

def read_csv_robust(path: str | Path, sep: str = ";", dtype=str) -> pd.DataFrame:
    """
    Lectura robusta para CSVs con posibles problemas de encoding.
    """
    attempts = [
        {"sep": sep, "encoding": "utf-8", "engine": "c"},
        {"sep": sep, "encoding": "latin-1", "engine": "c"},
        {"sep": sep, "encoding": "cp1252", "engine": "c"},
        {"sep": None, "encoding": "latin-1", "engine": "python"},
    ]

    last_error = None
    for cfg in attempts:
        try:
            kwargs = {
                "filepath_or_buffer": path,
                "dtype": dtype,
            }

            if cfg["engine"] == "c":
                kwargs.update(
                    sep=cfg["sep"],
                    encoding=cfg["encoding"],
                    engine="c",
                    low_memory=False,
                    na_values='-',
                )
            else:
                kwargs.update(
                    sep=cfg["sep"],
                    encoding=cfg["encoding"],
                    engine="python",
                    na_values='-',
                )

            df = pd.read_csv(**kwargs)
            if df.shape[1] <= 1:
                continue
            return df

        except Exception as e:
            last_error = e

    raise RuntimeError(f"No se pudo leer {path}: {last_error}")


def resolve_adatrap_stage_layout(
    columns: list[str],
    *,
    stage_layout_yaml: Optional[str | Path] = None,
) -> dict[str, list[str]]:
    if stage_layout_yaml is not None:
        layout = load_yaml_file(stage_layout_yaml)
        return {
            "trip_level": list(layout.get("trip_level", [])),
            "stage_1": list(layout.get("stage_1", [])),
            "stage_2": list(layout.get("stage_2", [])),
            "stage_3": list(layout.get("stage_3", [])),
            "stage_4": list(layout.get("stage_4", [])),
        }

    raise ValueError(
        "Para ADATRAP en este preprocess se espera stage_layout_yaml explícito. "
        "No se recomienda inferirlo aquí."
    )


# -----------------------------------------------------------------------------
# Helpers de nulos / strings
# -----------------------------------------------------------------------------

def is_missing_like(value: Any) -> bool:
    if pd.isna(value):
        return True
    s = str(value).strip()
    return s.lower() in NULL_LIKE_VALUES


def first_valid(*values: Any) -> Any:
    for v in values:
        if not is_missing_like(v):
            return v
    return pd.NA


def last_valid(*values: Any) -> Any:
    for v in reversed(values):
        if not is_missing_like(v):
            return v
    return pd.NA


# -----------------------------------------------------------------------------
# Helpers de normalización de paraderos / estaciones
# -----------------------------------------------------------------------------

def fix_station(station: Any) -> Any:
    if is_missing_like(station):
        return pd.NA

    station = str(station).strip()
    result_list = station.split("-")

    if len(result_list) < 2:
        return station

    if not result_list[-1].isdigit():
        result_list[-1], result_list[-2] = result_list[-2], result_list[-1]
        return "-".join(result_list)

    return station


def standardize_spanish_station(text: Any) -> Any:
    if is_missing_like(text):
        return pd.NA

    text = str(text).strip()
    parts = text.split("-")

    if len(parts) < 2:
        normalized_text = unicodedata.normalize("NFKD", text)
        standardized_text = "".join(
            c for c in normalized_text if not unicodedata.combining(c)
        )
        standardized_text = standardized_text.replace("`", "").upper().strip()
        return standardized_text

    return text


def normalize_stop_code(value: Any) -> Any:
    if is_missing_like(value):
        return pd.NA

    v = fix_station(value)
    v = standardize_spanish_station(v)

    if pd.isna(v):
        return pd.NA

    return str(v).strip().upper()


# -----------------------------------------------------------------------------
# Helpers de coordenadas
# -----------------------------------------------------------------------------

def parse_number_series(series: pd.Series) -> pd.Series:
    return pd.to_numeric(
        series.astype(str).str.replace(",", ".", regex=False),
        errors="coerce",
    )


def xy_to_lonlat(
    x: pd.Series,
    y: pd.Series,
    *,
    source_crs: str = DEFAULT_SOURCE_CRS,
    target_crs: str = DEFAULT_TARGET_CRS,
) -> tuple[pd.Series, pd.Series]:
    """
    Convierte series X/Y proyectadas a lon/lat WGS84.
    """
    x_num = parse_number_series(x)
    y_num = parse_number_series(y)

    valid = x_num.notna() & y_num.notna()

    lon = pd.Series(np.nan, index=x.index, dtype="float64")
    lat = pd.Series(np.nan, index=x.index, dtype="float64")

    if valid.any():
        transformer = Transformer.from_crs(source_crs, target_crs, always_xy=True)
        lon_vals, lat_vals = transformer.transform(
            x_num.loc[valid].to_numpy(),
            y_num.loc[valid].to_numpy(),
        )
        lon.loc[valid] = lon_vals
        lat.loc[valid] = lat_vals

    return lon, lat


# -----------------------------------------------------------------------------
# Tabla de paraderos / estaciones
# -----------------------------------------------------------------------------

def prepare_adatrap_stops_table(
    df_stops: pd.DataFrame,
    *,
    stop_code_col: str = "parada/est.metro",
    x_col: str = "x",
    y_col: str = "y",
) -> pd.DataFrame:
    stops = df_stops.copy()

    if stop_code_col not in stops.columns:
        raise KeyError(f"No existe la columna {stop_code_col!r} en df_stops")

    stops["stop_norm"] = stops[stop_code_col].map(normalize_stop_code)

    keep_cols = [c for c in [stop_code_col, x_col, y_col, "stop_norm"] if c in stops.columns]
    stops = stops[keep_cols].copy()

    stops = stops.dropna(subset=["stop_norm"]).drop_duplicates(subset=["stop_norm"])
    return stops


def lookup_stop_xy(
    stop_series: pd.Series,
    stops_df: pd.DataFrame,
    *,
    x_col: str = "x",
    y_col: str = "y",
) -> tuple[pd.Series, pd.Series]:
    """
    Busca x/y para una serie de códigos de parada/estación.
    No deja columnas auxiliares en el dataframe final.
    """
    stop_norm = stop_series.map(normalize_stop_code)
    stop_indexed = stops_df.set_index("stop_norm")

    x = stop_norm.map(stop_indexed[x_col]) if x_col in stop_indexed.columns else pd.Series(np.nan, index=stop_series.index)
    y = stop_norm.map(stop_indexed[y_col]) if y_col in stop_indexed.columns else pd.Series(np.nan, index=stop_series.index)

    return x, y


# -----------------------------------------------------------------------------
# Helpers de layout / etapas
# -----------------------------------------------------------------------------

def infer_stage_count_from_row(row: pd.Series) -> int:
    """
    Usa 'etapas' si existe; si no, detecta la última etapa presente.
    """
    if "etapas" in row.index:
        try:
            val = int(float(str(row["etapas"]).strip()))
            if 0 <= val <= 4:
                return val
        except Exception:
            pass

    stage_presence = 0
    for i, suf in enumerate(["1era", "2da", "3era", "4ta"], start=1):
        possible = [
            row.get(f"paraderosubida_{suf}"),
            row.get(f"paraderobajada_{suf}"),
            row.get(f"tipotransporte_{suf}"),
            row.get(f"t_{suf}_etapa"),
        ]
        if any(not is_missing_like(v) for v in possible):
            stage_presence = i

    return stage_presence


def strip_stage_suffix(col: str) -> str:
    """
    Normaliza nombres stage-specific a una versión genérica de etapa.
    Ejemplos:
    - t_1era_etapa -> t_etapa
    - paraderosubida_2da -> paraderosubida_etapa
    - linea_metro_subida_3 -> linea_metro_subida_etapa
    """
    col = re.sub(r"_(1era|2da|3era|4ta)_etapa$", "_etapa", col)
    col = re.sub(r"_(1era|2da|3era|4ta)$", "_etapa", col)
    col = re.sub(r"_(1|2|3|4)$", "_etapa", col)
    return col


def to_trip_summary_name(col: str) -> str:
    """
    Convierte un nombre stage-specific a un nombre de viaje resumido.
    Ejemplos:
    - paraderosubida_1era -> paraderosubida
    - tiempobajada_4ta -> tiempobajada
    - linea_metro_subida_1 -> linea_metro_subida
    """
    out = strip_stage_suffix(col)
    out = re.sub(r"_etapa$", "", out)
    return out


def is_start_related_stage_column(col: str) -> bool:
    """
    Campos que representan el inicio/subida de una etapa.
    """
    return "subida" in col.lower()


def is_end_related_stage_column(col: str) -> bool:
    """
    Campos que representan el término/bajada de una etapa.
    """
    return "bajada" in col.lower()


# -----------------------------------------------------------------------------
# Caso 1: viajes resumidos ADATRAP
# -----------------------------------------------------------------------------

def prepare_adatrap_triplevel_for_import(
    df_viajes: pd.DataFrame,
    df_stops: pd.DataFrame,
    *,
    stage_layout_yaml: str | Path,
    stop_code_col: str = "parada/est.metro",
    x_col: str = "x",
    y_col: str = "y",
    source_crs: str = DEFAULT_SOURCE_CRS,
    target_crs: str = DEFAULT_TARGET_CRS,
) -> pd.DataFrame:
    """
    Preprocess manual nivel 1 para ADATRAP viajes resumidos.

    Devuelve:
    - solo campos comunes del viaje (trip_level del YAML),
    - más campos relacionados a la subida/inicio de la primera etapa,
    - más campos relacionados a la bajada/término de la última etapa,
    - más coordenadas DD finales:
        * subida_lon, subida_lat
        * bajada_lon, bajada_lat

    NO deja:
    - columnas de detalle de otras etapas,
    - columnas auxiliares *_norm,
    - coordenadas X/Y auxiliares,
    - H3,
    - campos renombrados a canon Golondrina.
    """
    df = df_viajes.copy()
    stops = prepare_adatrap_stops_table(
        df_stops,
        stop_code_col=stop_code_col,
        x_col=x_col,
        y_col=y_col,
    )
    layout = resolve_adatrap_stage_layout(
        df.columns.tolist(),
        stage_layout_yaml=stage_layout_yaml,
    )

    trip_level_cols = [c for c in layout["trip_level"] if c in df.columns]

    stage_cols_by_stage = {
        1: [c for c in layout["stage_1"] if c in df.columns],
        2: [c for c in layout["stage_2"] if c in df.columns],
        3: [c for c in layout["stage_3"] if c in df.columns],
        4: [c for c in layout["stage_4"] if c in df.columns],
    }

    start_bases = {}
    end_bases = {}

    for stage_num, cols in stage_cols_by_stage.items():
        for col in cols:
            if is_start_related_stage_column(col):
                base = to_trip_summary_name(col)
                start_bases.setdefault(base, []).append((stage_num, col))
            if is_end_related_stage_column(col):
                base = to_trip_summary_name(col)
                end_bases.setdefault(base, []).append((stage_num, col))

    rows = []

    for _, row in df.iterrows():
        out = {}

        # campos comunes del viaje
        for c in trip_level_cols:
            out[c] = row.get(c)

        stage_count = infer_stage_count_from_row(row)
        out["etapas_detectadas"] = stage_count

        # inicio/subida de la primera etapa real
        for base, candidates in start_bases.items():
            ordered_cols = [col for _, col in sorted(candidates, key=lambda t: t[0])]
            out[base] = first_valid(*[row.get(c) for c in ordered_cols])

        # término/bajada de la última etapa real
        for base, candidates in end_bases.items():
            ordered_cols = [col for _, col in sorted(candidates, key=lambda t: t[0])]
            out[base] = last_valid(*[row.get(c) for c in ordered_cols])

        rows.append(out)

    trips_df = pd.DataFrame(rows)

    # Coordenadas DD desde la tabla de paraderos
    if "paraderosubida" in trips_df.columns:
        sx, sy = lookup_stop_xy(trips_df["paraderosubida"], stops, x_col=x_col, y_col=y_col)
        subida_lon, subida_lat = xy_to_lonlat(
            sx, sy,
            source_crs=source_crs,
            target_crs=target_crs,
        )
        trips_df["subida_lon"] = subida_lon
        trips_df["subida_lat"] = subida_lat

    if "paraderobajada" in trips_df.columns:
        bx, by = lookup_stop_xy(trips_df["paraderobajada"], stops, x_col=x_col, y_col=y_col)
        bajada_lon, bajada_lat = xy_to_lonlat(
            bx, by,
            source_crs=source_crs,
            target_crs=target_crs,
        )
        trips_df["bajada_lon"] = bajada_lon
        trips_df["bajada_lat"] = bajada_lat

    return trips_df


# -----------------------------------------------------------------------------
# Caso 2: stages ADATRAP -> long
# -----------------------------------------------------------------------------

def prepare_adatrap_stagelevel_for_import(
    df_viajes: pd.DataFrame,
    df_stops: pd.DataFrame,
    *,
    stage_layout_yaml: str | Path,
    stop_code_col: str = "parada/est.metro",
    x_col: str = "x",
    y_col: str = "y",
    source_crs: str = DEFAULT_SOURCE_CRS,
    target_crs: str = DEFAULT_TARGET_CRS,
) -> pd.DataFrame:
    """
    Preprocess manual nivel 1 para ADATRAP stages.

    Devuelve:
    - campos comunes del viaje,
    - más campos de la etapa correspondiente, normalizados a nombres
      terminados en '_etapa',
    - más coordenadas DD:
        * subida_lon, subida_lat
        * bajada_lon, bajada_lat

    Ya NO deja columnas del tipo:
    - tespera_3era_etapa
    - tipotransporte_2da
    etc.
    Sino:
    - tespera_etapa
    - tipotransporte_etapa
    - paraderosubida_etapa
    - paraderobajada_etapa
    etc.
    """
    df = df_viajes.copy()
    stops = prepare_adatrap_stops_table(
        df_stops,
        stop_code_col=stop_code_col,
        x_col=x_col,
        y_col=y_col,
    )
    layout = resolve_adatrap_stage_layout(
        df.columns.tolist(),
        stage_layout_yaml=stage_layout_yaml,
    )

    trip_level_cols = [c for c in layout["trip_level"] if c in df.columns]
    stage_cols_by_stage = {
        1: [c for c in layout["stage_1"] if c in df.columns],
        2: [c for c in layout["stage_2"] if c in df.columns],
        3: [c for c in layout["stage_3"] if c in df.columns],
        4: [c for c in layout["stage_4"] if c in df.columns],
    }

    rows = []

    for _, row in df.iterrows():
        stage_count = infer_stage_count_from_row(row)
        if stage_count <= 0:
            continue

        for stage_num in range(1, stage_count + 1):
            stage_cols = stage_cols_by_stage.get(stage_num, [])

            # si la etapa no tiene contenido real, no crear fila
            stage_values = [row.get(c) for c in stage_cols]
            if not any(not is_missing_like(v) for v in stage_values):
                continue

            out = {}

            # campos comunes del viaje
            for c in trip_level_cols:
                out[c] = row.get(c)

            # número de etapa
            out["numero_etapa"] = stage_num

            # campos de esta etapa
            for c in stage_cols:
                out[strip_stage_suffix(c)] = row.get(c)

            rows.append(out)

    stages_df = pd.DataFrame(rows)

    if stages_df.empty:
        return stages_df

    # Coordenadas DD desde tabla de paraderos
    if "paraderosubida_etapa" in stages_df.columns:
        sx, sy = lookup_stop_xy(stages_df["paraderosubida_etapa"], stops, x_col=x_col, y_col=y_col)
        subida_lon, subida_lat = xy_to_lonlat(
            sx, sy,
            source_crs=source_crs,
            target_crs=target_crs,
        )
        stages_df["subida_lon"] = subida_lon
        stages_df["subida_lat"] = subida_lat

    if "paraderobajada_etapa" in stages_df.columns:
        bx, by = lookup_stop_xy(stages_df["paraderobajada_etapa"], stops, x_col=x_col, y_col=y_col)
        bajada_lon, bajada_lat = xy_to_lonlat(
            bx, by,
            source_crs=source_crs,
            target_crs=target_crs,
        )
        stages_df["bajada_lon"] = bajada_lon
        stages_df["bajada_lat"] = bajada_lat

    return stages_df

In [108]:
adatrap_viajes = read_csv_robust(DATA_PATH / "semana_2019_05_13.viajes", sep="|")
adatrap_stops = read_csv_robust(DATA_PATH / "DIC_777_fixed.csv", sep=",")

In [109]:
adatrap_sample = adatrap_viajes.sample(50000).sort_index()
df_adatrap_trips_for_import = prepare_adatrap_triplevel_for_import(
    df_viajes=adatrap_sample,
    df_stops=adatrap_stops,
    stage_layout_yaml=DATA_PATH / "stage_layout.yaml",
)

In [110]:
#adatrap_trips_sample = df_adatrap_trips_for_import.sample(5000).sort_index()

In [111]:
display(df_adatrap_trips_for_import)

,id,nviaje,netapa,etapas,netapassinbajada,ultimaetapaconbajada,tviaje_seg,tviaje_min,dviajeeuclidiana_mts,dviajeenruta_mts,paraderosubida,paraderobajada,comunasubida,comunabajada,diseno777subida,diseno777bajada,tiemposubida,tiempobajada,periodosubida,periodobajada,tipodia,mediahora,contrato,factorexpansion,tiempomediodeviaje,periodomediodeviaje,mediahoramediodeviaje,tipodiamediodeviaje,escolar,tviaje_en_vehiculo_min,tipo_corte_etapa_viaje,proposito,dviaje_buses,etapas_detectadas,linea_metro_subida,zona777subida,linea_metro_bajada,zona777bajada,subida_lon,subida_lat,bajada_lon,bajada_lat
0,18836274,1,2,NaN,0,1,4571.0000,76.1833,19561.5859,30047.0000,T-13-279-PO-5,INES DE SUAREZ,MAIPU,PROVIDENCIA,741,183,2019-05-13 07:32:20,2019-05-13 08:48:31,04 - PUNTA MANANA,05 - TRANSICION PUNTA MANANA,LABORAL,07:30:00,NaN,1.3822,2019-05-13 08:10:25,04 - PUNTA MANANA,08:00:00,LABORAL,NaN,74.5833,SM2H,TRABAJO,NaN,2,L5,741,L6,183,-70.794655,-33.519780,NaN,NaN
1,18838306,2,1,NaN,0,1,1223.0000,20.3833,5799.6108,7105.0000,MANUEL MONTT,RONDIZONNI,PROVIDENCIA,SANTIAGO,175,303,2019-05-13 17:34:13,2019-05-13 17:54:36,09 - PUNTA TARDE,09 - PUNTA TARDE,LABORAL,17:30:00,NaN,1.2629,2019-05-13 17:44:24,09 - PUNTA TARDE,17:30:00,LABORAL,NaN,20.3833,UE,HOGAR,NaN,1,L1,175,L2,303,-70.619214,-33.428346,-70.655929,-33.470625
2,18871970,2,2,NaN,1,1,4636.0000,77.2667,10142.3428,NaN,T-26-228-NS-40,LOS HEROES,LA CISTERNA,SANTIAGO,379,282,2019-05-13 16:52:53,2019-05-13 18:03:13,08 - FUERA DE PUNTA TARDE,09 - PUNTA TARDE,LABORAL,16:30:00,NaN,2.1078,2019-05-13 17:24:35,08 - FUERA DE PUNTA TARDE,17:00:00,LABORAL,NaN,19.5000,SM2H,TRABAJO,NaN,2,L2,379,L2,282,-70.664489,-33.537804,-70.660505,-33.446476
3,18873122,1,2,NaN,0,1,2740.0000,45.6667,10437.1836,12574.0000,L-32-23-35-OP,T-20-177-OP-8,PENALOLEN,SANTIAGO,462,282,2019-05-13 11:05:12,2019-05-13 11:50:52,06 - FUERA DE PUNTA MANANA,06 - FUERA DE PUNTA MANANA,LABORAL,11:00:00,NaN,1.3238,2019-05-13 11:28:02,06 - FUERA DE PUNTA MANANA,11:00:00,LABORAL,NaN,42.0667,SM2H,TRABAJO,NaN,2,<NA>,462,<NA>,282,-70.557515,-33.486403,-70.663834,-33.456875
4,18883954,2,1,NaN,0,1,1243.0000,20.7167,8466.1377,8591.0000,LOS LEONES L1,SAN ALBERTO HURTADO,PROVIDENCIA,ESTACION CENTRAL,179,77,2019-05-13 11:45:20,2019-05-13 12:06:03,06 - FUERA DE PUNTA MANANA,06 - FUERA DE PUNTA MANANA,LABORAL,11:30:00,NaN,1.1798,2019-05-13 11:55:41,06 - FUERA DE PUNTA MANANA,11:30:00,LABORAL,NaN,20.7167,MS_M_M,OTROS,NaN,1,L1,179,L1,77,NaN,NaN,-70.692442,-33.454116
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49995,4197925220,1,2,NaN,1,0,NaN,NaN,NaN,NaN,UNIVERSIDAD DE SANTIAGO,SANTA ANA,ESTACION CENTRAL,NaN,78,NaN,2019-05-19 17:10:05,2019-05-19 17:22:48,05 - MEDIODIA DOMINGO,NaN,DOMINGO,17:00:00,NaN,0.0000,NaN,NaN,NaN,DOMINGO,NaN,12.7167,UE,SINBAJADA,NaN,2,L1,78,L2,274,-70.686157,-33.452903,-70.660112,-33.438311
49996,4199316916,2,1,NaN,0,1,1914.0000,31.9000,10922.5781,11284.0000,L-32-11-45-NS,T-20-189-OP-15,PENALOLEN,SANTIAGO,765,292,2019-05-19 16:17:37,2019-05-19 16:49:31,05 - MEDIODIA DOMINGO,05 - MEDIODIA DOMINGO,DOMINGO,16:00:00,NaN,1.5775,2019-05-19 16:33:34,05 - MEDIODIA DOMINGO,16:30:00,DOMINGO,NaN,31.9000,UE,HOGAR,NaN,1,<NA>,765,<NA>,292,-70.524725,-33.479289,-70.639388,-33.457345
49997,4249639891,1,1,NaN,1,0,NaN,NaN,NaN,NaN,MACUL,<NA>,MACUL,NaN,465,NaN,2019-05-19 13:07:25,<NA>,04 - MANANA DOMINGO,NaN,DOMINGO,13:00:00,NaN,0.0000,NaN,NaN,NaN,DOMINGO,NaN,0.0000,UE,SINBAJADA,NaN,1,L4,465,<NA>,<NA>,-70.589491,-33.508795,NaN,NaN
49998,4254427651,1,1,NaN,0,1,980.0000,16.3333,6776.3511,7265.0000,NEPTUNO,UNIVERSIDAD DE CHILE,LO PRADO,SANTIAGO,119,285,2019-05-19 10:25:39,2019-05-19 10:41:59,04 - MANANA DOMINGO,04 - MANANA DOMINGO,DOMINGO,10:00:00,NaN,1.4021,2019-05-19 10:33:49,04 - MANANA DOMINGO,10:30:00,DOMINGO,NaN,16.3333,SM2H,TRABAJO,NaN,1,L1,119,L1,285,-70.722697,-33.451553,-70.650398,-33.443942


In [112]:
df_adatrap_stages_for_import = prepare_adatrap_stagelevel_for_import(
    df_viajes=adatrap_sample,
    df_stops=adatrap_stops,
    stage_layout_yaml=DATA_PATH / "stage_layout.yaml",
)

In [113]:
display(df_adatrap_stages_for_import)

,id,nviaje,netapa,etapas,netapassinbajada,ultimaetapaconbajada,tviaje_seg,tviaje_min,dviajeeuclidiana_mts,dviajeenruta_mts,paraderosubida,paraderobajada,comunasubida,comunabajada,diseno777subida,diseno777bajada,tiemposubida,tiempobajada,periodosubida,periodobajada,tipodia,mediahora,contrato,factorexpansion,tiempomediodeviaje,periodomediodeviaje,mediahoramediodeviaje,tipodiamediodeviaje,escolar,tviaje_en_vehiculo_min,tipo_corte_etapa_viaje,proposito,dviaje_buses,numero_etapa,t_etapa,d_etapa,tespera_etapa,ttrasbordo_etapa,tcaminata_etapa,dtransbordo_etapa,op_etapa,tipoop_etapa,serv_etapa,linea_metro_subida_etapa,linea_metro_bajada_etapa,paraderosubida_etapa,tiemposubida_etapa,zona777subida_etapa,paraderobajada_etapa,tiempobajada_etapa,zona777bajada_etapa,tipotransporte_etapa,tesperaest_etapa,subida_lon,subida_lat,bajada_lon,bajada_lat
0,18836274,1,2,NaN,0,1,4571.0000,76.1833,19561.5859,30047.0000,T-13-279-PO-5,INES DE SUAREZ,MAIPU,PROVIDENCIA,741,183,2019-05-13 07:32:20,2019-05-13 08:48:31,04 - PUNTA MANANA,05 - TRANSICION PUNTA MANANA,LABORAL,07:30:00,NaN,1.3822,2019-05-13 08:10:25,04 - PUNTA MANANA,08:00:00,LABORAL,NaN,74.5833,SM2H,TRABAJO,NaN,1,24.0333,4245.0000,0.4243,1.6000,1.1757,105.8159,4,NaN,T405 00I,NaN,NaN,T-13-279-PO-5,2019-05-13 07:32:20,741,E-13-54-SN-10,2019-05-13 07:56:22,819,BUS,NaN,-70.794655,-33.519780,-70.757084,-33.509308
1,18836274,1,2,NaN,0,1,4571.0000,76.1833,19561.5859,30047.0000,T-13-279-PO-5,INES DE SUAREZ,MAIPU,PROVIDENCIA,741,183,2019-05-13 07:32:20,2019-05-13 08:48:31,04 - PUNTA MANANA,05 - TRANSICION PUNTA MANANA,LABORAL,07:30:00,NaN,1.3822,2019-05-13 08:10:25,04 - PUNTA MANANA,08:00:00,LABORAL,NaN,74.5833,SM2H,TRABAJO,NaN,2,50.5500,25802.0000,NaN,NaN,NaN,NaN,NaN,NaN,L5,L5,L6,PLAZA MAIPU,2019-05-13 07:57:58,819,INES DE SUAREZ,2019-05-13 08:48:31,183,METRO,NaN,-70.756241,-33.510222,NaN,NaN
2,18838306,2,1,NaN,0,1,1223.0000,20.3833,5799.6108,7105.0000,MANUEL MONTT,RONDIZONNI,PROVIDENCIA,SANTIAGO,175,303,2019-05-13 17:34:13,2019-05-13 17:54:36,09 - PUNTA TARDE,09 - PUNTA TARDE,LABORAL,17:30:00,NaN,1.2629,2019-05-13 17:44:24,09 - PUNTA TARDE,17:30:00,LABORAL,NaN,20.3833,UE,HOGAR,NaN,1,20.3833,7105.0000,NaN,NaN,NaN,NaN,NaN,NaN,L1,L1,L2,MANUEL MONTT,2019-05-13 17:34:13,175,RONDIZONNI,2019-05-13 17:54:36,303,METRO,NaN,-70.619214,-33.428346,-70.655929,-33.470625
3,18871970,2,2,NaN,1,1,4636.0000,77.2667,10142.3428,NaN,T-26-228-NS-40,LOS HEROES,LA CISTERNA,SANTIAGO,379,282,2019-05-13 16:45:57,2019-05-13 18:03:13,08 - FUERA DE PUNTA TARDE,09 - PUNTA TARDE,LABORAL,16:30:00,NaN,2.1078,2019-05-13 17:24:35,08 - FUERA DE PUNTA TARDE,17:00:00,LABORAL,NaN,19.5000,SM2H,TRABAJO,NaN,1,NaN,0.0000,NaN,NaN,NaN,NaN,2,NaN,T248 00R,NaN,NaN,T-26-228-NS-40,2019-05-13 16:52:53,379,NaN,NaN,NaN,ZP,NaN,-70.664489,-33.537804,NaN,NaN
4,18871970,2,2,NaN,1,1,4636.0000,77.2667,10142.3428,NaN,T-26-228-NS-40,LOS HEROES,LA CISTERNA,SANTIAGO,379,282,2019-05-13 16:45:57,2019-05-13 18:03:13,08 - FUERA DE PUNTA TARDE,09 - PUNTA TARDE,LABORAL,16:30:00,NaN,2.1078,2019-05-13 17:24:35,08 - FUERA DE PUNTA TARDE,17:00:00,LABORAL,NaN,19.5000,SM2H,TRABAJO,NaN,2,19.5000,10669.0000,NaN,NaN,NaN,NaN,NaN,NaN,L2,L2,L2,LA CISTERNA L2,2019-05-13 17:43:43,380,LOS HEROES,2019-05-13 18:03:13,282,METRO,NaN,NaN,NaN,-70.660505,-33.446476
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67234,4197925220,1,2,NaN,1,0,NaN,NaN,NaN,NaN,UNIVERSIDAD DE SANTIAGO,NaN,ESTACION CENTRAL,NaN,78,NaN,2019-05-19 17:10:05,NaN,05 - MEDIODIA DOMINGO,NaN,DOMINGO,17:00:00,NaN,0.0000,NaN,NaN,NaN,DOMINGO,NaN,12.7167,UE,SINBAJADA,NaN,2,NaN,0.0000,NaN,NaN,NaN,NaN,3,NaN,T303 00R,NaN,NaN,T-20-73-OP-50,2019-05-19 17:38:09,275,NaN,NaN,NaN,BUS,NaN,-70.658969,-33.437001,NaN,NaN
67235,4199316916,2,1,NaN,0,1,1914.0000,31.9000,10922.5781,11284.0000,L-32-11-45-NS,T-20-189-OP-15,PENALOLEN,SANTIAGO,765,292,2019-05-19 16:17

## Ahora probamos el import

In [114]:
from pylondrina.importing import ImportOptions, import_trips_from_dataframe
from pylondrina.schema import TripSchema, FieldSpec, DomainSpec, TripSchemaEffective
from pylondrina.datasets import TripDataset
from pylondrina.reports import ImportReport, Issue
from pylondrina.errors import ImportError as PylondrinaImportError, SchemaError

### Helpers

In [115]:
from __future__ import annotations

from pathlib import Path
from typing import Any

from pylondrina.importing import ImportOptions
from pylondrina.schema import TripSchema, FieldSpec, DomainSpec


def load_yaml_file(path: str | Path) -> dict[str, Any]:
    import yaml
    with open(path, "r", encoding="utf-8") as f:
        return yaml.safe_load(f) or {}


def clean_domain_values(values: list[Any]) -> list[str]:
    """
    Limpia un dominio YAML:
    - convierte a str
    - hace strip
    - elimina vacíos
    - deduplica preservando orden
    """
    out = []
    seen = set()

    for v in values:
        s = str(v).strip()
        if not s:
            continue
        if s not in seen:
            seen.add(s)
            out.append(s)

    return out


def clean_domain_dict(domains_raw: dict[str, list[Any]]) -> dict[str, list[str]]:
    return {
        k: clean_domain_values(v)
        for k, v in domains_raw.items()
    }


def merge_domains(domains: dict[str, list[str]], *keys: str) -> list[str]:
    merged = []
    seen = set()

    for key in keys:
        for v in domains.get(key, []):
            if v not in seen:
                seen.add(v)
                merged.append(v)

    return merged


def build_adatrap_time_period_mapping(domain_values: list[str]) -> dict[str, str]:
    """
    Mapea los periodos largos ADATRAP a time_period canónico Golondrina.
    """
    mapping = {}

    for v in domain_values:
        s = v.upper().strip()

        if "PRE NOCTURNO" in s:
            mapping[v] = "evening"
        elif "NOCTURNO" in s:
            mapping[v] = "night"
        elif "MANANA" in s:
            mapping[v] = "morning"
        elif "MEDIODIA" in s:
            mapping[v] = "midday"
        elif "TARDE" in s:
            mapping[v] = "afternoon"

    return mapping

### Schema base

In [116]:
BASE_GOLONDRINA_IMPORT_SCHEMA = TripSchema(
    version="1.1",
    fields={
        # required core
        "movement_id": FieldSpec("movement_id", "string", required=True),
        "user_id": FieldSpec("user_id", "string", required=True),
        "origin_longitude": FieldSpec("origin_longitude", "float", required=True),
        "origin_latitude": FieldSpec("origin_latitude", "float", required=True),
        "destination_longitude": FieldSpec("destination_longitude", "float", required=True),
        "destination_latitude": FieldSpec("destination_latitude", "float", required=True),
        "origin_h3_index": FieldSpec("origin_h3_index", "string", required=True),
        "destination_h3_index": FieldSpec("destination_h3_index", "string", required=True),
        "origin_time_utc": FieldSpec("origin_time_utc", "datetime", required=False),
        "destination_time_utc": FieldSpec("destination_time_utc", "datetime", required=False),
        "trip_id": FieldSpec("trip_id", "string", required=True),
        "movement_seq": FieldSpec("movement_seq", "int", required=True),

        # base categorical
        "mode": FieldSpec(
            "mode",
            "categorical",
            required=False,
            domain=DomainSpec(
                values=[
                    "walk", "bicycle", "scooter", "motorcycle", "car",
                    "taxi", "ride_hailing", "bus", "metro", "train", "other"
                ],
                extendable=True,
            ),
        ),
        "purpose": FieldSpec(
            "purpose",
            "categorical",
            required=False,
            domain=DomainSpec(
                values=[
                    "home", "work", "education", "shopping",
                    "errand", "health", "leisure", "transfer", "other"
                ],
                extendable=True,
            ),
        ),
        "day_type": FieldSpec(
            "day_type",
            "categorical",
            required=False,
            domain=DomainSpec(
                values=["weekday", "weekend", "holiday"],
                extendable=True,
            ),
        ),
        "time_period": FieldSpec(
            "time_period",
            "categorical",
            required=False,
            domain=DomainSpec(
                values=["night", "morning", "midday", "afternoon", "evening"],
                extendable=True,
            ),
        ),
        "user_gender": FieldSpec(
            "user_gender",
            "categorical",
            required=False,
            domain=DomainSpec(
                values=["female", "male", "other", "unknown"],
                extendable=True,
            ),
        ),
        "user_age_group": FieldSpec(
            "user_age_group",
            "categorical",
            required=False,
            domain=DomainSpec(
                values=["0-14", "15-24", "25-34", "35-44", "45-54", "55-64", "65-plus", "unknown"],
                extendable=True,
            ),
        ),
        "income_quintile": FieldSpec(
            "income_quintile",
            "categorical",
            required=False,
            domain=DomainSpec(
                values=["1", "2", "3", "4", "5", "unknown"],
                extendable=True,
            ),
        ),

        # base non-categorical
        "origin_time_local_hhmm": FieldSpec("origin_time_local_hhmm", "string", required=False),
        "destination_time_local_hhmm": FieldSpec("destination_time_local_hhmm", "string", required=False),
        "origin_municipality": FieldSpec("origin_municipality", "string", required=False),
        "destination_municipality": FieldSpec("destination_municipality", "string", required=False),
        "trip_weight": FieldSpec("trip_weight", "float", required=False),
        "mode_sequence": FieldSpec("mode_sequence", "string", required=False),
    },
    required=[
        "movement_id",
        "user_id",
        "origin_longitude",
        "origin_latitude",
        "destination_longitude",
        "destination_latitude",
        "origin_h3_index",
        "destination_h3_index",
        "trip_id",
        "movement_seq",
    ],
)

### Importamos trips

In [117]:
def make_adatrap_trips_import_schema(adatrap_domains_yaml: str | Path) -> TripSchema:
    domains_raw = load_yaml_file(adatrap_domains_yaml)
    domains = clean_domain_dict(domains_raw)

    extra_fields = {
        "ultimaetapaconbajada": FieldSpec(
            "ultimaetapaconbajada",
            "categorical",
            required=False,
            domain=DomainSpec(values=domains["ultimaetapaconbajada"], extendable=True),
        ),
        "tipodiamediodeviaje": FieldSpec(
            "tipodiamediodeviaje",
            "categorical",
            required=False,
            domain=DomainSpec(values=domains["tipodiamediodeviaje"], extendable=True),
        ),
        "tipo_corte_etapa_viaje": FieldSpec(
            "tipo_corte_etapa_viaje",
            "categorical",
            required=False,
            domain=DomainSpec(values=domains["tipo_corte_etapa_viaje"], extendable=True),
        ),
        "periodosubida": FieldSpec(
            "periodosubida",
            "categorical",
            required=False,
            domain=DomainSpec(values=domains["periodosubida"], extendable=True),
        ),
        "periodobajada": FieldSpec(
            "periodobajada",
            "categorical",
            required=False,
            domain=DomainSpec(values=domains["periodobajada"], extendable=True),
        ),
        "periodomediodeviaje": FieldSpec(
            "periodomediodeviaje",
            "categorical",
            required=False,
            domain=DomainSpec(values=domains["periodomediodeviaje"], extendable=True),
        ),
        "mediahora": FieldSpec(
            "mediahora",
            "categorical",
            required=False,
            domain=DomainSpec(values=domains["mediahora"], extendable=True),
        ),
        "mediahoramediodeviaje": FieldSpec(
            "mediahoramediodeviaje",
            "categorical",
            required=False,
            domain=DomainSpec(values=domains["mediahoramediodeviaje"], extendable=True),
        ),
        "linea_metro_subida": FieldSpec(
            "linea_metro_subida",
            "categorical",
            required=False,
            domain=DomainSpec(
                values=merge_domains(
                    domains,
                    "linea_metro_subida_1",
                    "linea_metro_subida_2",
                    "linea_metro_subida_3",
                    "linea_metro_subida_4",
                ),
                extendable=True,
            ),
        ),
        "linea_metro_bajada": FieldSpec(
            "linea_metro_bajada",
            "categorical",
            required=False,
            domain=DomainSpec(
                values=merge_domains(
                    domains,
                    "linea_metro_bajada_1",
                    "linea_metro_bajada_2",
                    "linea_metro_bajada_3",
                    "linea_metro_bajada_4",
                ),
                extendable=True,
            ),
        ),
    }
    
    return TripSchema(
        version=BASE_GOLONDRINA_IMPORT_SCHEMA.version,
        fields={**BASE_GOLONDRINA_IMPORT_SCHEMA.fields, **extra_fields},
        required=list(BASE_GOLONDRINA_IMPORT_SCHEMA.required),
    )

In [118]:
ADATRAP_TRIPS_IMPORT_OPTIONS = ImportOptions(
    keep_extra_fields=True,
    selected_fields=None,
    strict=False,
    strict_domains=False,
    single_stage=True,
    source_timezone="America/Santiago",
)

In [119]:
ADATRAP_TRIPS_FIELD_CORRESPONDENCE = {
    "user_id": "id",

    "origin_longitude": "subida_lon",
    "origin_latitude": "subida_lat",
    "destination_longitude": "bajada_lon",
    "destination_latitude": "bajada_lat",

    "origin_time_utc": "tiemposubida",
    "destination_time_utc": "tiempobajada",

    "origin_municipality": "comunasubida",
    "destination_municipality": "comunabajada",

    "purpose": "proposito",
    "day_type": "tipodia",
    "time_period": "periodomediodeviaje",

    "trip_weight": "factorexpansion",
}

In [120]:
def make_adatrap_trips_value_correspondence(adatrap_domains_yaml: str | Path) -> dict[str, dict[str, str]]:
    domains_raw = load_yaml_file(adatrap_domains_yaml)
    domains = clean_domain_dict(domains_raw)

    return {
        "purpose": {
            "HOGAR": "home",
            "TRABAJO": "work",
            "OTROS": "other",
            "MENOS1MINUTO": "other",
            "SINBAJADA": "other",
        },
        "day_type": {
            "LABORAL": "weekday",
            "SABADO": "weekend",
            "DOMINGO": "weekend",
        },
        "time_period": build_adatrap_time_period_mapping(domains["periodomediodeviaje"]),
    }

In [121]:
ADATRAP_TRIPS_SCHEMA = make_adatrap_trips_import_schema(DATA_PATH / "domains.yaml")
ADATRAP_TRIPS_OPTIONS = ADATRAP_TRIPS_IMPORT_OPTIONS
ADATRAP_TRIPS_FIELD_CORR = ADATRAP_TRIPS_FIELD_CORRESPONDENCE
ADATRAP_TRIPS_VALUE_CORR = make_adatrap_trips_value_correspondence(DATA_PATH / "domains.yaml")

In [122]:
trips_adatrap, report_adatrap = import_trips_from_dataframe(
    df_adatrap_trips_for_import,
    schema=ADATRAP_TRIPS_SCHEMA,
    source_name="ADATRAP trips level-1 manual preprocess",
    options=ADATRAP_TRIPS_OPTIONS,
    field_correspondence=ADATRAP_TRIPS_FIELD_CORR,
    value_correspondence=ADATRAP_TRIPS_VALUE_CORR,
    provenance={
        "source": {
            "name": "ADATRAP",
            "entity": "trips",
            "version": "perfil_semana",
        },
        "notes": ["preprocess manual nivel 1 en notebook"],
    },
    h3_resolution=8,
)

In [123]:
display(trips_adatrap.data)

,movement_id,trip_id,movement_seq,user_id,nviaje,netapa,etapas,netapassinbajada,ultimaetapaconbajada,tviaje_seg,tviaje_min,dviajeeuclidiana_mts,dviajeenruta_mts,paraderosubida,paraderobajada,origin_municipality,destination_municipality,diseno777subida,diseno777bajada,origin_time_utc,destination_time_utc,periodosubida,periodobajada,day_type,mediahora,contrato,trip_weight,tiempomediodeviaje,time_period,mediahoramediodeviaje,tipodiamediodeviaje,escolar,tviaje_en_vehiculo_min,tipo_corte_etapa_viaje,purpose,dviaje_buses,etapas_detectadas,linea_metro_subida,zona777subida,linea_metro_bajada,zona777bajada,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_h3_index,destination_h3_index
0,m0,m0,0,18836274,1,2,NaN,0,1,4571.0000,76.1833,19561.5859,30047.0000,T-13-279-PO-5,INES DE SUAREZ,MAIPU,PROVIDENCIA,741,183,2019-05-13 11:32:20+00:00,2019-05-13 12:48:31+00:00,04 - PUNTA MANANA,05 - TRANSICION PUNTA MANANA,weekday,07:30:00,NaN,1.3822,2019-05-13 08:10:25,morning,08:00:00,LABORAL,NaN,74.5833,SM2H,work,NaN,2,L5,741,L6,183,-70.794655,-33.519780,NaN,NaN,88b2c540c1fffff,<NA>
1,m1,m1,0,18838306,2,1,NaN,0,1,1223.0000,20.3833,5799.6108,7105.0000,MANUEL MONTT,RONDIZONNI,PROVIDENCIA,SANTIAGO,175,303,2019-05-13 21:34:13+00:00,2019-05-13 21:54:36+00:00,09 - PUNTA TARDE,09 - PUNTA TARDE,weekday,17:30:00,NaN,1.2629,2019-05-13 17:44:24,afternoon,17:30:00,LABORAL,NaN,20.3833,UE,home,NaN,1,L1,175,L2,303,-70.619214,-33.428346,-70.655929,-33.470625,88b2c556abfffff,88b2c554a9fffff
2,m2,m2,0,18871970,2,2,NaN,1,1,4636.0000,77.2667,10142.3428,NaN,T-26-228-NS-40,LOS HEROES,LA CISTERNA,SANTIAGO,379,282,2019-05-13 20:52:53+00:00,2019-05-13 22:03:13+00:00,08 - FUERA DE PUNTA TARDE,09 - PUNTA TARDE,weekday,16:30:00,NaN,2.1078,2019-05-13 17:24:35,afternoon,17:00:00,LABORAL,NaN,19.5000,SM2H,work,NaN,2,L2,379,L2,282,-70.664489,-33.537804,-70.660505,-33.446476,88b2c54631fffff,88b2c55417fffff
3,m3,m3,0,18873122,1,2,NaN,0,1,2740.0000,45.6667,10437.1836,12574.0000,L-32-23-35-OP,T-20-177-OP-8,PENALOLEN,SANTIAGO,462,282,2019-05-13 15:05:12+00:00,2019-05-13 15:50:52+00:00,06 - FUERA DE PUNTA MANANA,06 - FUERA DE PUNTA MANANA,weekday,11:00:00,NaN,1.3238,2019-05-13 11:28:02,morning,11:00:00,LABORAL,NaN,42.0667,SM2H,work,NaN,2,<NA>,462,<NA>,282,-70.557515,-33.486403,-70.663834,-33.456875,88b2c50865fffff,88b2c554edfffff
4,m4,m4,0,18883954,2,1,NaN,0,1,1243.0000,20.7167,8466.1377,8591.0000,LOS LEONES L1,SAN ALBERTO HURTADO,PROVIDENCIA,ESTACION CENTRAL,179,77,2019-05-13 15:45:20+00:00,2019-05-13 16:06:03+00:00,06 - FUERA DE PUNTA MANANA,06 - FUERA DE PUNTA MANANA,weekday,11:30:00,NaN,1.1798,2019-05-13 11:55:41,morning,11:30:00,LABORAL,NaN,20.7167,MS_M_M,other,NaN,1,L1,179,L1,77,NaN,NaN,-70.692442,-33.454116,<NA>,88b2c555cbfffff
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49995,m49995,m49995,0,4197925220,1,2,NaN,1,0,NaN,NaN,NaN,NaN,UNIVERSIDAD DE SANTIAGO,SANTA ANA,ESTACION CENTRAL,<NA>,78,NaN,2019-05-19 21:10:05+00:00,2019-05-19 21:22:48+00:00,05 - MEDIODIA DOMINGO,<NA>,weekend,17:00:00,NaN,0.0000,NaN,<NA>,<NA>,DOMINGO,NaN,12.7167,UE,other,NaN,2,L1,78,L2,274,-70.686157,-33.452903,-70.660112,-33.438311,88b2c55435fffff,88b2c55411fffff
49996,m49996,m49996,0,4199316916,2,1,NaN,0,1,1914.0000,31.9000,10922.5781,11284.0000,L-32-11-45-NS,T-20-189-OP-15,PENALOLEN,SANTIAGO,765,292,2019-05-19 20:17:37+00:00,2019-05-19 20:49:31+00:00,05 - MEDIODIA DOMINGO,05 - MEDIODIA DOMINGO,weekend,16:00:00,NaN,1.5775,2019-05-19 16:33:34,midday,16:30:00,DOMINGO,NaN,31.9000,UE,home,NaN,1,<NA>,765,<NA>,292,-70.524725,-33.479289,-70.639388,-33.457345,88b2c50843fffff,88b2c554c7fffff
49997,m49997,m49997,0,4249639891,1,1,NaN,1,0,NaN,NaN,NaN,NaN,MACUL,<NA>,MACUL,<NA>,465,NaN,2019-05-19 17:07:25+00:00,NaT,04 - MANANA DOMINGO,<NA>,weekend,13:00:00,NaN,0.0000,NaN,<NA>,<NA>,DOMINGO,NaN,0.0000,UE,other,NaN,1,L4,465,<NA>,<NA>,-70.589491,-33.5

In [124]:
display(report_adatrap.issues)

[Issue(level='info', code='IMP.INPUT.OPTIONAL_FIELD_NOT_FOUND', message="El campo opcional 'mode' no está presente en la fuente y no se encontró correspondencia; se omitirá.", field='mode', source_field=None, row_count=None, details={'field': 'mode', 'source_columns': ['id', 'nviaje', 'netapa', 'etapas', 'netapassinbajada', 'ultimaetapaconbajada', 'tviaje_seg', 'tviaje_min', 'dviajeeuclidiana_mts', 'dviajeenruta_mts', 'paraderosubida', 'paraderobajada', 'comunasubida', 'comunabajada', 'diseno777subida', 'diseno777bajada', 'tiemposubida', 'tiempobajada', 'periodosubida', 'periodobajada', 'tipodia', 'mediahora', 'contrato', 'factorexpansion', 'tiempomediodeviaje', 'periodomediodeviaje', 'mediahoramediodeviaje', 'tipodiamediodeviaje', 'escolar', 'tviaje_en_vehiculo_min', 'tipo_corte_etapa_viaje', 'proposito', 'dviaje_buses', 'etapas_detectadas', 'linea_metro_subida', 'zona777subida', 'linea_metro_bajada', 'zona777bajada', 'subida_lon', 'subida_lat', 'bajada_lon', 'bajada_lat'], 'field_cor

### Importamos stages

In [125]:
def make_adatrap_stages_import_schema(adatrap_domains_yaml: str | Path) -> TripSchema:
    domains_raw = load_yaml_file(adatrap_domains_yaml)
    domains = clean_domain_dict(domains_raw)

    extra_fields = {
        "ultimaetapaconbajada": FieldSpec(
            "ultimaetapaconbajada",
            "categorical",
            required=False,
            domain=DomainSpec(values=domains["ultimaetapaconbajada"], extendable=True),
        ),
        "tipodiamediodeviaje": FieldSpec(
            "tipodiamediodeviaje",
            "categorical",
            required=False,
            domain=DomainSpec(values=domains["tipodiamediodeviaje"], extendable=True),
        ),
        "tipo_corte_etapa_viaje": FieldSpec(
            "tipo_corte_etapa_viaje",
            "categorical",
            required=False,
            domain=DomainSpec(values=domains["tipo_corte_etapa_viaje"], extendable=True),
        ),
        "periodosubida": FieldSpec(
            "periodosubida",
            "categorical",
            required=False,
            domain=DomainSpec(values=domains["periodosubida"], extendable=True),
        ),
        "periodobajada": FieldSpec(
            "periodobajada",
            "categorical",
            required=False,
            domain=DomainSpec(values=domains["periodobajada"], extendable=True),
        ),
        "periodomediodeviaje": FieldSpec(
            "periodomediodeviaje",
            "categorical",
            required=False,
            domain=DomainSpec(values=domains["periodomediodeviaje"], extendable=True),
        ),
        "mediahora": FieldSpec(
            "mediahora",
            "categorical",
            required=False,
            domain=DomainSpec(values=domains["mediahora"], extendable=True),
        ),
        "mediahoramediodeviaje": FieldSpec(
            "mediahoramediodeviaje",
            "categorical",
            required=False,
            domain=DomainSpec(values=domains["mediahoramediodeviaje"], extendable=True),
        ),
        "op_etapa": FieldSpec(
            "op_etapa",
            "categorical",
            required=False,
            domain=DomainSpec(
                values=merge_domains(
                    domains,
                    "op_1era_etapa",
                    "op_2da_etapa",
                    "op_3era_etapa",
                    "op_4ta_etapa",
                ),
                extendable=True,
            ),
        ),
        "linea_metro_subida_etapa": FieldSpec(
            "linea_metro_subida_etapa",
            "categorical",
            required=False,
            domain=DomainSpec(
                values=merge_domains(
                    domains,
                    "linea_metro_subida_1",
                    "linea_metro_subida_2",
                    "linea_metro_subida_3",
                    "linea_metro_subida_4",
                ),
                extendable=True,
            ),
        ),
        "linea_metro_bajada_etapa": FieldSpec(
            "linea_metro_bajada_etapa",
            "categorical",
            required=False,
            domain=DomainSpec(
                values=merge_domains(
                    domains,
                    "linea_metro_bajada_1",
                    "linea_metro_bajada_2",
                    "linea_metro_bajada_3",
                    "linea_metro_bajada_4",
                ),
                extendable=True,
            ),
        ),
    }

    stage_fields = {f: fs for f, fs in BASE_GOLONDRINA_IMPORT_SCHEMA.fields.items() if f != "trip_id"}
    stage_required = [r for r in BASE_GOLONDRINA_IMPORT_SCHEMA.required if r != "trip_id"]

    return TripSchema(
        version=BASE_GOLONDRINA_IMPORT_SCHEMA.version,
        fields={**stage_fields, **extra_fields},
        required=list(stage_required),
    )

In [126]:
ADATRAP_STAGES_IMPORT_OPTIONS = ImportOptions(
    keep_extra_fields=True,
    selected_fields=None,
    strict=False,
    strict_domains=False,
    single_stage=False,
    source_timezone="America/Santiago",
)

In [127]:
ADATRAP_STAGES_FIELD_CORRESPONDENCE = {
    "user_id": "id",
    "movement_seq": "numero_etapa",

    "origin_longitude": "subida_lon",
    "origin_latitude": "subida_lat",
    "destination_longitude": "bajada_lon",
    "destination_latitude": "bajada_lat",

    "origin_time_utc": "tiemposubida_etapa",
    "destination_time_utc": "tiempobajada_etapa",

    "origin_municipality": "comunasubida",
    "destination_municipality": "comunabajada",

    "mode": "tipotransporte_etapa",
    "purpose": "proposito",
    "day_type": "tipodia",
    "time_period": "periodomediodeviaje",

    "trip_weight": "factorexpansion",
}

In [128]:
def make_adatrap_stages_value_correspondence(adatrap_domains_yaml: str | Path) -> dict[str, dict[str, str]]:
    domains_raw = load_yaml_file(adatrap_domains_yaml)
    domains = clean_domain_dict(domains_raw)

    return {
        "mode": {
            "BUS": "bus",
            "METRO": "metro",
            "METROTREN": "train",
            "ZP": "other",
        },
        "purpose": {
            "HOGAR": "home",
            "TRABAJO": "work",
            "OTROS": "other",
            "MENOS1MINUTO": "other",
            "SINBAJADA": "other",
        },
        "day_type": {
            "LABORAL": "weekday",
            "SABADO": "weekend",
            "DOMINGO": "weekend",
        },
        "time_period": build_adatrap_time_period_mapping(domains["periodomediodeviaje"]),
    }

In [129]:
ADATRAP_STAGES_SCHEMA = make_adatrap_stages_import_schema(DATA_PATH / "domains.yaml")
ADATRAP_STAGES_OPTIONS = ADATRAP_STAGES_IMPORT_OPTIONS
ADATRAP_STAGES_FIELD_CORR = ADATRAP_STAGES_FIELD_CORRESPONDENCE
ADATRAP_STAGES_VALUE_CORR = make_adatrap_stages_value_correspondence(DATA_PATH / "domains.yaml")

In [130]:
stages_adatrap, report_stages_adatrap = import_trips_from_dataframe(
    df_adatrap_stages_for_import,
    schema=ADATRAP_STAGES_SCHEMA,
    source_name="ADATRAP stages level-1 manual preprocess",
    options=ADATRAP_STAGES_OPTIONS,
    field_correspondence=ADATRAP_STAGES_FIELD_CORR,
    value_correspondence=ADATRAP_STAGES_VALUE_CORR,
    provenance={
        "source": {
            "name": "ADATRAP",
            "entity": "stages",
            "version": "perfil_semana",
        },
        "notes": ["preprocess manual nivel 1 en notebook"],
    },
    h3_resolution=8,
)

In [131]:
display(stages_adatrap.data)

,movement_id,user_id,nviaje,netapa,etapas,netapassinbajada,ultimaetapaconbajada,tviaje_seg,tviaje_min,dviajeeuclidiana_mts,dviajeenruta_mts,paraderosubida,paraderobajada,origin_municipality,destination_municipality,diseno777subida,diseno777bajada,tiemposubida,tiempobajada,periodosubida,periodobajada,day_type,mediahora,contrato,trip_weight,tiempomediodeviaje,time_period,mediahoramediodeviaje,tipodiamediodeviaje,escolar,tviaje_en_vehiculo_min,tipo_corte_etapa_viaje,purpose,dviaje_buses,movement_seq,t_etapa,d_etapa,tespera_etapa,ttrasbordo_etapa,tcaminata_etapa,dtransbordo_etapa,op_etapa,tipoop_etapa,serv_etapa,linea_metro_subida_etapa,linea_metro_bajada_etapa,paraderosubida_etapa,origin_time_utc,zona777subida_etapa,paraderobajada_etapa,destination_time_utc,zona777bajada_etapa,mode,tesperaest_etapa,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_h3_index,destination_h3_index
0,m0,18836274,1,2,NaN,0,1,4571.0000,76.1833,19561.5859,30047.0000,T-13-279-PO-5,INES DE SUAREZ,MAIPU,PROVIDENCIA,741,183,2019-05-13 07:32:20,2019-05-13 08:48:31,04 - PUNTA MANANA,05 - TRANSICION PUNTA MANANA,weekday,07:30:00,NaN,1.3822,2019-05-13 08:10:25,morning,08:00:00,LABORAL,NaN,74.5833,SM2H,work,NaN,1,24.0333,4245.0000,0.4243,1.6000,1.1757,105.8159,4,NaN,T405 00I,<NA>,<NA>,T-13-279-PO-5,2019-05-13 11:32:20+00:00,741,E-13-54-SN-10,2019-05-13 11:56:22+00:00,819,bus,NaN,-70.794655,-33.519780,-70.757084,-33.509308,88b2c540c1fffff,88b2c542bbfffff
1,m1,18836274,1,2,NaN,0,1,4571.0000,76.1833,19561.5859,30047.0000,T-13-279-PO-5,INES DE SUAREZ,MAIPU,PROVIDENCIA,741,183,2019-05-13 07:32:20,2019-05-13 08:48:31,04 - PUNTA MANANA,05 - TRANSICION PUNTA MANANA,weekday,07:30:00,NaN,1.3822,2019-05-13 08:10:25,morning,08:00:00,LABORAL,NaN,74.5833,SM2H,work,NaN,2,50.5500,25802.0000,NaN,NaN,NaN,NaN,<NA>,NaN,L5,L5,L6,PLAZA MAIPU,2019-05-13 11:57:58+00:00,819,INES DE SUAREZ,2019-05-13 12:48:31+00:00,183,metro,NaN,-70.756241,-33.510222,NaN,NaN,88b2c54297fffff,<NA>
2,m2,18838306,2,1,NaN,0,1,1223.0000,20.3833,5799.6108,7105.0000,MANUEL MONTT,RONDIZONNI,PROVIDENCIA,SANTIAGO,175,303,2019-05-13 17:34:13,2019-05-13 17:54:36,09 - PUNTA TARDE,09 - PUNTA TARDE,weekday,17:30:00,NaN,1.2629,2019-05-13 17:44:24,afternoon,17:30:00,LABORAL,NaN,20.3833,UE,home,NaN,1,20.3833,7105.0000,NaN,NaN,NaN,NaN,<NA>,NaN,L1,L1,L2,MANUEL MONTT,2019-05-13 21:34:13+00:00,175,RONDIZONNI,2019-05-13 21:54:36+00:00,303,metro,NaN,-70.619214,-33.428346,-70.655929,-33.470625,88b2c556abfffff,88b2c554a9fffff
3,m3,18871970,2,2,NaN,1,1,4636.0000,77.2667,10142.3428,NaN,T-26-228-NS-40,LOS HEROES,LA CISTERNA,SANTIAGO,379,282,2019-05-13 16:45:57,2019-05-13 18:03:13,08 - FUERA DE PUNTA TARDE,09 - PUNTA TARDE,weekday,16:30:00,NaN,2.1078,2019-05-13 17:24:35,afternoon,17:00:00,LABORAL,NaN,19.5000,SM2H,work,NaN,1,NaN,0.0000,NaN,NaN,NaN,NaN,2,NaN,T248 00R,<NA>,<NA>,T-26-228-NS-40,2019-05-13 20:52:53+00:00,379,NaN,NaT,NaN,other,NaN,-70.664489,-33.537804,NaN,NaN,88b2c54631fffff,<NA>
4,m4,18871970,2,2,NaN,1,1,4636.0000,77.2667,10142.3428,NaN,T-26-228-NS-40,LOS HEROES,LA CISTERNA,SANTIAGO,379,282,2019-05-13 16:45:57,2019-05-13 18:03:13,08 - FUERA DE PUNTA TARDE,09 - PUNTA TARDE,weekday,16:30:00,NaN,2.1078,2019-05-13 17:24:35,afternoon,17:00:00,LABORAL,NaN,19.5000,SM2H,work,NaN,2,19.5000,10669.0000,NaN,NaN,NaN,NaN,<NA>,NaN,L2,L2,L2,LA CISTERNA L2,2019-05-13 21:43:43+00:00,380,LOS HEROES,2019-05-13 22:03:13+00:00,282,metro,NaN,NaN,NaN,-70.660505,-33.446476,<NA>,88b2c55417fffff
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67234,m67234,4197925220,1,2,NaN,1,0,NaN,NaN,NaN,NaN,UNIVERSIDAD DE SANTIAGO,NaN,ESTACION CENTRAL,<NA>,78,NaN,2019-05-19 17:10:05,NaN,05 - MEDIODIA DOMINGO,<NA>,weekend,17:00:00,NaN,0.0000,NaN,<NA>,<NA>,DOMINGO,NaN,12.7167,UE,other,NaN,2,NaN,0.0000,NaN,NaN,NaN,NaN,3,NaN,T303 00R,<NA>,<NA>

In [132]:
display(report_stages_adatrap.issues)

[Issue(level='info', code='IMP.INPUT.OPTIONAL_FIELD_NOT_FOUND', message="El campo opcional 'user_gender' no está presente en la fuente y no se encontró correspondencia; se omitirá.", field='user_gender', source_field=None, row_count=None, details={'field': 'user_gender', 'source_columns': ['id', 'nviaje', 'netapa', 'etapas', 'netapassinbajada', 'ultimaetapaconbajada', 'tviaje_seg', 'tviaje_min', 'dviajeeuclidiana_mts', 'dviajeenruta_mts', 'paraderosubida', 'paraderobajada', 'comunasubida', 'comunabajada', 'diseno777subida', 'diseno777bajada', 'tiemposubida', 'tiempobajada', 'periodosubida', 'periodobajada', 'tipodia', 'mediahora', 'contrato', 'factorexpansion', 'tiempomediodeviaje', 'periodomediodeviaje', 'mediahoramediodeviaje', 'tipodiamediodeviaje', 'escolar', 'tviaje_en_vehiculo_min', 'tipo_corte_etapa_viaje', 'proposito', 'dviaje_buses', 'numero_etapa', 't_etapa', 'd_etapa', 'tespera_etapa', 'ttrasbordo_etapa', 'tcaminata_etapa', 'dtransbordo_etapa', 'op_etapa', 'tipoop_etapa', 's

# Nivel 3

## Para datos del sistema ADATRAP - Trips / viajes

### 1) Uso normal, rápido, sin decisiones de schema/campos/valores en la factory

In [3]:
from pylondrina.sources.helpers import import_trips_from_profile
from scripts.source_profiles.factories_adatrap.make_adatrap_trips_profile import make_adatrap_trips_profile
from scripts.source_profiles.factories_adatrap.trips_defaults import (
    ADATRAP_TRIPS_DEFAULT_PROVENANCE_EXAMPLE,
)

adatrap_viajes = pd.read_csv(
    DATA_PATH / "semana_2019_05_13.viajes",
    sep="|",
    dtype=str,
    low_memory=False,
    na_values="-"
)
adatrap_stops = pd.read_csv(
    DATA_PATH / "DIC_777_fixed.csv",
    sep=",",
    dtype=str,
    low_memory=False,
)

profile = make_adatrap_trips_profile(
    df_stops=adatrap_stops,
    stage_layout_yaml=DATA_PATH / "stage_layout.yaml",
    domains_yaml=DATA_PATH / "domains.yaml",
)

# inspección clara
#profile.schema_override
#profile.default_options
#profile.default_field_correspondence
#profile.default_value_correspondence

In [9]:
display(profile.schema_override)

{'version': '1.1',
 'required': ['movement_id',
              'user_id',
              'origin_longitude',
              'origin_latitude',
              'destination_longitude',
              'destination_latitude',
              'origin_h3_index',
              'destination_h3_index',
              'trip_id',
              'movement_seq'],
 'semantic_rules': None,
 'fields': {'movement_id': {'name': 'movement_id',
                            'dtype': 'string',
                            'required': True,
                            'constraints': None,
                            'domain': None},
            'user_id': {'name': 'user_id', 'dtype': 'string', 'required': True, 'constraints': None, 'domain': None},
            'origin_longitude': {'name': 'origin_longitude',
                                 'dtype': 'float',
                                 'required': True,
                                 'constraints': None,
                                 'domain': None},
            'origin_latitude': {'name': 'origin_latitude',
                                'dtype': 'float',
                                'required': True,
                                'constraints': None,
                                'domain': None},
            'destination_longitude': {'name': 'destination_longitude',
                                      'dtype': 'float',
                                      'required': True,
                                      'constraints': None,
                                      'domain': None},
            'destination_latitude': {'name': 'destination_latitude',
                                     'dtype': 'float',
                                     'required': True,
                                     'constraints': None,
                                     'domain': None},
            'origin_h3_index': {'name': 'origin_h3_index',
                                'dtype': 'string',
                                'required': True,
                                'constraints': None,
                                'domain': None},
            'destination_h3_index': {'name': 'destination_h3_index',
                                     'dtype': 'string',
                                     'required': True,
                                     'constraints': None,
                                     'domain': None},
            'origin_time_utc': {'name': 'origin_time_utc',
                                'dtype': 'datetime',
                                'required': False,
                                'constraints': None,
                                'domain': None},
            'destination_time_utc': {'name': 'destination_time_utc',
                                     'dtype': 'datetime',
                                     'required': False,
                                     'constraints': None,
                                     'domain': None},
            'trip_id': {'name': 'trip_id', 'dtype': 'string', 'required': True, 'constraints': None, 'domain': None},
            'movement_seq': {'name': 'movement_seq',
                             'dtype': 'int',
                             'required': True,
                             'constraints': None,
                             'domain': None},
            'mode': {'name': 'mode',
                     'dtype': 'categorical',
                     'required': False,
                     'constraints': None,
                     'domain': {'values': ['walk',
                                           'bicycle',
                                           'scooter',
                                           'motorcycle',
                                           'car',
                                           'taxi',
                                           'ride_hailing',
                                           'bus',
                                           'metro',
                                           '

In [4]:
trips_adatrap, report_adatrap = import_trips_from_profile(
    profile=profile,
    df=adatrap_viajes.head(5000),
    source_name="ADATRAP trips level-3 factory",
    provenance=ADATRAP_TRIPS_DEFAULT_PROVENANCE_EXAMPLE,
    h3_resolution=8,
)

display(trips_adatrap.data.head())
display(report_adatrap.issues)

,movement_id,trip_id,movement_seq,user_id,nviaje,netapa,etapas,netapassinbajada,ultimaetapaconbajada,tviaje_seg,tviaje_min,dviajeeuclidiana_mts,dviajeenruta_mts,origin_stop_code,destination_stop_code,origin_municipality,destination_municipality,diseno777subida,diseno777bajada,origin_time_utc,destination_time_utc,periodosubida,periodobajada,day_type,mediahora,contrato,trip_weight,tiempomediodeviaje,time_period,mediahoramediodeviaje,tipodiamediodeviaje,escolar,tviaje_en_vehiculo_min,tipo_corte_etapa_viaje,purpose,dviaje_buses,etapas_detectadas,linea_metro_subida,zona777subida,linea_metro_bajada,zona777bajada,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_h3_index,destination_h3_index
0,m0,m0,0,4306298,1,2,NaN,0,1,3098.0000,51.6333,15836.0068,25695.0000,T-13-104-PO-15,NUBLE,MAIPU,NUNOA,172,299,2019-05-13 10:00:55+00:00,2019-05-13 10:52:33+00:00,03 - TRANSICION NOCTURNO,04 - PUNTA MANANA,weekday,06:00:00,NaN,1.4376,2019-05-13 06:26:44,night,06:00:00,LABORAL,NaN,48.8667,SM2H,work,NaN,2,L5,172,L5,299,-70.783226,-33.524734,-70.626419,-33.467472,88b2c540ddfffff,88b2c5548bfffff
1,m1,m1,0,8398026,2,1,NaN,0,1,701.0000,11.6833,3872.8728,4084.0000,T-18-117-NS-5,T-18-153-PO-35,NUNOA,NUNOA,231,240,2019-05-13 10:53:45+00:00,2019-05-13 11:05:26+00:00,04 - PUNTA MANANA,04 - PUNTA MANANA,weekday,06:30:00,NaN,1.2547,2019-05-13 06:59:35,morning,06:30:00,LABORAL,NaN,11.6833,SM2H,work,NaN,1,<NA>,231,<NA>,240,-70.629251,-33.452086,-70.587011,-33.457846,88b2c554c3fffff,88b2c50b29fffff
2,m2,m2,0,9173978,1,1,NaN,0,1,690.0000,11.5000,4511.1367,4513.0000,SANTA LUCIA,SAN ALBERTO HURTADO,SANTIAGO,ESTACION CENTRAL,286,77,2019-05-13 22:12:35+00:00,2019-05-13 22:24:05+00:00,09 - PUNTA TARDE,09 - PUNTA TARDE,weekday,18:00:00,NaN,1.2976,2019-05-13 18:18:20,afternoon,18:00:00,LABORAL,NaN,11.5000,MS_M_M,other,NaN,1,L1,286,L1,77,-70.645817,-33.442849,-70.692442,-33.454116,88b2c55413fffff,88b2c555cbfffff
3,m3,m3,0,9284970,1,1,NaN,0,1,1209.0000,20.1500,3875.6917,5234.0000,T-17-149-SN-8,T-17-140-OP-35,LAS CONDES,LAS CONDES,775,207,2019-05-13 10:56:49+00:00,2019-05-13 11:16:58+00:00,04 - PUNTA MANANA,04 - PUNTA MANANA,weekday,06:30:00,NaN,1.4500,2019-05-13 07:06:53,morning,07:00:00,LABORAL,NaN,20.1500,SM2H,work,NaN,1,<NA>,775,<NA>,207,-70.529608,-33.420266,-70.569059,-33.409114,88b2c51995fffff,88b2c519adfffff
4,m4,m4,0,9403002,1,1,NaN,0,1,1320.0000,22.0000,9128.2852,9258.0000,ESCUELA MILITAR,UNION LATINO AMERICANA,LAS CONDES,SANTIAGO,215,280,2019-05-13 20:08:12+00:00,2019-05-13 20:30:12+00:00,08 - FUERA DE PUNTA TARDE,08 - FUERA DE PUNTA TARDE,weekday,16:00:00,NaN,1.2666,2019-05-13 16:19:12,afternoon,16:00:00,LABORAL,NaN,22.0000,M3B,other,NaN,1,L1,215,L1,280,-70.584445,-33.414320,-70.673213,-33.449471,88b2c556d3fffff,88b2c5543bfffff


[Issue(level='info', code='IMP.INPUT.OPTIONAL_FIELD_NOT_FOUND', message="El campo opcional 'mode' no está presente en la fuente y no se encontró correspondencia; se omitirá.", field='mode', source_field=None, row_count=None, details={'field': 'mode', 'source_columns': ['id', 'nviaje', 'netapa', 'etapas', 'netapassinbajada', 'ultimaetapaconbajada', 'tviaje_seg', 'tviaje_min', 'dviajeeuclidiana_mts', 'dviajeenruta_mts', 'paraderosubida', 'paraderobajada', 'comunasubida', 'comunabajada', 'diseno777subida', 'diseno777bajada', 'tiemposubida', 'tiempobajada', 'periodosubida', 'periodobajada', 'tipodia', 'mediahora', 'contrato', 'factorexpansion', 'tiempomediodeviaje', 'periodomediodeviaje', 'mediahoramediodeviaje', 'tipodiamediodeviaje', 'escolar', 'tviaje_en_vehiculo_min', 'tipo_corte_etapa_viaje', 'proposito', 'dviaje_buses', 'etapas_detectadas', 'linea_metro_subida', 'zona777subida', 'linea_metro_bajada', 'zona777bajada', 'subida_lon', 'subida_lat', 'bajada_lon', 'bajada_lat'], 'field_cor

### 2) Uso más personalizado, con decisiones específicas

In [5]:
from scripts.source_profiles.factories_adatrap.trips_defaults import (
    make_adatrap_trips_default_schema,
    make_adatrap_trips_default_value_correspondence,
    load_yaml_file,
    clean_domain_dict,
    build_adatrap_time_period_mapping,
)
from scripts.source_profiles.factories_adatrap.make_adatrap_trips_profile import _merge_field_correspondence
from pylondrina.importing import ImportOptions
from pylondrina.schema import TripSchema, FieldSpec, DomainSpec
from pylondrina.types import FieldCorrespondence, ValueCorrespondence

# -----------------------------------------------------------------------------
# Objetos para caso personalizado
# -----------------------------------------------------------------------------

def make_adatrap_trips_custom_schema(adatrap_domains_yaml: str | Path) -> TripSchema:
    base = make_adatrap_trips_default_schema(adatrap_domains_yaml)

    selected = {
        k: v
        for k, v in base.fields.items()
        if k in {
            "movement_id",
            "user_id",
            "origin_longitude",
            "origin_latitude",
            "destination_longitude",
            "destination_latitude",
            "origin_h3_index",
            "destination_h3_index",
            "trip_id",
            "movement_seq",
            "origin_time_utc",
            "destination_time_utc",
            "origin_municipality",
            "destination_municipality",
            "origin_stop_code",
            "destination_stop_code",
            "purpose",
            "day_type",
            "time_period",
            "trip_weight",
        }
    }

    return TripSchema(
        version="1.1-adatrap-custom",
        fields=selected,
        required=[
            "movement_id",
            "user_id",
            "origin_longitude",
            "origin_latitude",
            "destination_longitude",
            "destination_latitude",
            "origin_h3_index",
            "destination_h3_index",
            "trip_id",
            "movement_seq",
        ],
    )

ADATRAP_TRIPS_CUSTOM_OPTIONS = ImportOptions(
    keep_extra_fields=False,
    selected_fields=[
        "movement_id",
        "user_id",
        "origin_longitude",
        "origin_latitude",
        "destination_longitude",
        "destination_latitude",
        "origin_h3_index",
        "destination_h3_index",
        "trip_id",
        "movement_seq",
        "origin_time_utc",
        "destination_time_utc",
        "origin_municipality",
        "destination_municipality",
        "origin_stop_code",
        "destination_stop_code",
        "purpose",
        "day_type",
        "time_period",
        "trip_weight",
    ],
    strict=False,
    strict_domains=False,
    single_stage=True,
    source_timezone="America/Santiago",
)

ADATRAP_TRIPS_CUSTOM_FIELD_CORRESPONDENCE: FieldCorrespondence = {
    "user_id": "id",
    "origin_longitude": "subida_lon",
    "origin_latitude": "subida_lat",
    "destination_longitude": "bajada_lon",
    "destination_latitude": "bajada_lat",
    "origin_time_utc": "tiemposubida",
    "destination_time_utc": "tiempobajada",
    "origin_municipality": "comunasubida",
    "destination_municipality": "comunabajada",
    "origin_stop_code": "paraderosubida",
    "destination_stop_code": "paraderobajada",
    "purpose": "proposito",
    "day_type": "tipodia",
    "time_period": "periodosubida",
    "trip_weight": "factorexpansion",
}

def make_adatrap_trips_custom_value_correspondence(
    adatrap_domains_yaml: str | Path,
) -> ValueCorrespondence:
    base = make_adatrap_trips_default_value_correspondence(adatrap_domains_yaml)
    base["purpose"]["SINBAJADA"] = "other"

    domains_raw = load_yaml_file(adatrap_domains_yaml)
    domains = clean_domain_dict(domains_raw)
    base["time_period"] = build_adatrap_time_period_mapping(domains["periodosubida"])

    return base

ADATRAP_TRIPS_CUSTOM_PROVENANCE_EXAMPLE = {
    "source": {
        "name": "ADATRAP",
        "profile": "ADATRAP_TRIPS_CUSTOM",
        "entity": "trips",
        "version": "perfil_semana",
    },
    "notes": [
        "factory nivel 3 ADATRAP trips custom",
        "time_period usa periodosubida",
    ],
}

In [7]:
custom_schema = make_adatrap_trips_custom_schema(DATA_PATH / "domains.yaml")
custom_value_corr = make_adatrap_trips_custom_value_correspondence(DATA_PATH / "domains.yaml")

profile_custom = make_adatrap_trips_profile(
    df_stops=adatrap_stops,
    stage_layout_yaml=DATA_PATH / "stage_layout.yaml",
    domains_yaml=DATA_PATH / "domains.yaml",
    schema_override=custom_schema,
    value_correspondence_override=custom_value_corr,
    options_override={
        "keep_extra_fields": ADATRAP_TRIPS_CUSTOM_OPTIONS.keep_extra_fields,
        "selected_fields": ADATRAP_TRIPS_CUSTOM_OPTIONS.selected_fields,
        "strict": ADATRAP_TRIPS_CUSTOM_OPTIONS.strict,
        "strict_domains": ADATRAP_TRIPS_CUSTOM_OPTIONS.strict_domains,
        "single_stage": ADATRAP_TRIPS_CUSTOM_OPTIONS.single_stage,
        "source_timezone": ADATRAP_TRIPS_CUSTOM_OPTIONS.source_timezone,
    },
    profile_name="ADATRAP_TRIPS_CUSTOM",
)


# inspección clara
#profile_custom.schema_override
#profile_custom.default_options
display(profile_custom.default_field_correspondence)
#profile_custom.default_value_correspondence

{'user_id': 'id',
 'origin_longitude': 'subida_lon',
 'origin_latitude': 'subida_lat',
 'destination_longitude': 'bajada_lon',
 'destination_latitude': 'bajada_lat',
 'origin_time_utc': 'tiemposubida',
 'destination_time_utc': 'tiempobajada',
 'origin_municipality': 'comunasubida',
 'destination_municipality': 'comunabajada',
 'origin_stop_code': 'paraderosubida',
 'destination_stop_code': 'paraderobajada',
 'purpose': 'proposito',
 'day_type': 'tipodia',
 'time_period': 'periodomediodeviaje',
 'trip_weight': 'factorexpansion'}

In [8]:
trips_adatrap_custom, report_adatrap_custom = import_trips_from_profile(
    profile=profile_custom,
    df=adatrap_viajes.head(5000),
    field_correspondence=ADATRAP_TRIPS_CUSTOM_FIELD_CORRESPONDENCE,
    source_name="ADATRAP trips level-3 custom factory",
    provenance=ADATRAP_TRIPS_CUSTOM_PROVENANCE_EXAMPLE,
    h3_resolution=8,
)

display(trips_adatrap_custom.data.head())
display(report_adatrap_custom.issues)

,movement_id,trip_id,movement_seq,user_id,origin_stop_code,destination_stop_code,origin_municipality,destination_municipality,origin_time_utc,destination_time_utc,time_period,day_type,trip_weight,purpose,origin_longitude,origin_latitude,destination_longitude,destination_latitude,origin_h3_index,destination_h3_index
0,m0,m0,0,4306298,T-13-104-PO-15,NUBLE,MAIPU,NUNOA,2019-05-13 10:00:55+00:00,2019-05-13 10:52:33+00:00,night,weekday,1.4376,work,-70.783226,-33.524734,-70.626419,-33.467472,88b2c540ddfffff,88b2c5548bfffff
1,m1,m1,0,8398026,T-18-117-NS-5,T-18-153-PO-35,NUNOA,NUNOA,2019-05-13 10:53:45+00:00,2019-05-13 11:05:26+00:00,morning,weekday,1.2547,work,-70.629251,-33.452086,-70.587011,-33.457846,88b2c554c3fffff,88b2c50b29fffff
2,m2,m2,0,9173978,SANTA LUCIA,SAN ALBERTO HURTADO,SANTIAGO,ESTACION CENTRAL,2019-05-13 22:12:35+00:00,2019-05-13 22:24:05+00:00,afternoon,weekday,1.2976,other,-70.645817,-33.442849,-70.692442,-33.454116,88b2c55413fffff,88b2c555cbfffff
3,m3,m3,0,9284970,T-17-149-SN-8,T-17-140-OP-35,LAS CONDES,LAS CONDES,2019-05-13 10:56:49+00:00,2019-05-13 11:16:58+00:00,morning,weekday,1.4500,work,-70.529608,-33.420266,-70.569059,-33.409114,88b2c51995fffff,88b2c519adfffff
4,m4,m4,0,9403002,ESCUELA MILITAR,UNION LATINO AMERICANA,LAS CONDES,SANTIAGO,2019-05-13 20:08:12+00:00,2019-05-13 20:30:12+00:00,afternoon,weekday,1.2666,other,-70.584445,-33.414320,-70.673213,-33.449471,88b2c556d3fffff,88b2c5543bfffff


[Issue(level='warning', code='IMP.TYPE.COERCE_PARTIAL', message="La conversión mínima del campo 'destination_time_utc' falló en algunas filas (13.2%); se marcarán como nulos para validación posterior.", field='destination_time_utc', source_field=None, row_count=662, details={'field': 'destination_time_utc', 'dtype_expected': 'datetime', 'parse_fail_count': 662, 'total_count': 5000, 'fail_rate': 0.1324, 'fallback': 'set_null', 'action': 'continue'}),
 Issue(level='warning', code='IMP.TYPE.COERCE_PARTIAL', message="La conversión mínima del campo 'destination_time_utc' falló en algunas filas (13.2%); se marcarán como nulos para validación posterior.", field='destination_time_utc', source_field=None, row_count=662, details={'field': 'destination_time_utc', 'dtype_expected': 'datetime', 'parse_fail_count': 662, 'total_count': 5000, 'fail_rate': 0.1324, 'fallback': 'set_null', 'action': 'continue'}),
 Issue(level='warning', code='IMP.H3.PARTIAL_DERIVATION', message='La derivación de índices 

## Para datos del sistema ADATRAP - Stages / etapas

### 1) Uso normal, rápido, sin decisiones de schema/campos/valores en la factory

### 2) Uso más personalizado, con decisiones específicas